# 10 — `When2Speak` Closed-Loop Multi-Agent Collaboration: Turn-Taking Timing & Utterance Quality

**Self-Contained Publication Suite (Resolving Review Issues `M3`, `M4`, `M6`, `M7`, `M8`, `Z1`–`Z4`, `Z6`, `E6`)**:
1. **Equal Validation Calibration (`val_calib = 200` turns, Zero Leakage — Invariant #8)**: Every gating method (`B1 Single-Gate LLM/Jev`, `B2 ProACT`, `B0 Direct-SC@3`, `A4 Single-Combined`, `A4 Advocate-Only`, `A4 Skeptic-Only`, `DIAL-Parallel`, and **`DIAL-Collab` Closed-Loop Advocate→Skeptic Cross-Examination**) is calibrated on the exact same 200 validation turns maximizing Macro-F1, plus leak-free **Platt Scaling** for genuine `ECE`.
2. **Compute-Matched & Frontier Backbones**: Evaluates `gpt-4.1-mini`, `gpt-4.1`, frontier `gpt-5.6-terra`, and cross-family `DeepSeek-V4.1-Flash` / `GLM-5.3-flash`, reporting **uncached USD inference cost** (`uncached_cost_usd`).
3. **Rigorous Statistical Testing**: Paired Bootstrap $\Delta\text{Macro-F1}$ 95% CIs + $p$-values with **Holm–Bonferroni** family-wise error correction, plus full continuous **FIR–MIR Pareto frontier** against published When2Speak SFT/RL operating points.
4. **Utterance Quality Benchmark (`N = 100` Stratified `SPEAK` Turns)**: Directly tests **Stage-3 Draft–Audit–Refine** (`Executor` $\rightarrow$ `Pre-Delivery Skeptic-Auditor`) vs `Solo Executor` (Ablation `A8`) vs `Direct Chat (B0)` against human reference responses using a **Cross-Family Blinded Dual-Judge** (`gpt-4.1` + `DeepSeek-V4.1-Flash`).

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "datasets/pagemantle_hard",
    "caches",
    "runs/when2speak_v3_collab",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)

In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull), Install Dependencies & Hot-Reload ===
REPO_URL = os.environ.get("DIAL_SKILL_REPO_URL", "https://github.com/BoldHu/multiagent-speak.git")
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

import importlib, sys, types

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    try:
        import typesafe_sdk, duckdb, typer, tenacity, scipy, sklearn  # noqa: F401
        !pip install -q -e . --no-deps
    except ImportError:
        !pip install -q -e .

_src_dir = str((REPO_DIR / "src").resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")

In [ ]:
# === CELL 2: Bootstrap Multi-Provider Secrets & Configs (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import (
    describe_secret_sources,
    load_autodl_api_key,
    load_jev_api_key,
    load_openai_api_key,
)
from dial_skill.config import load_dial_skill_config, load_experiments_config, load_models_config
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role

print(
    "Secret lookup diagnostics:",
    json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT, include_autodl=True), indent=2),
)
OPENAI_API_KEY = load_openai_api_key(drive_root=DRIVE_ROOT, required=True)
JEV_API_KEY = load_jev_api_key(drive_root=DRIVE_ROOT, required=True)
AUTODL_API_KEY = load_autodl_api_key(drive_root=DRIVE_ROOT, required=False)
AUTODL_BASE_URL = os.environ.get("AUTODL_BASE_URL", "https://www.autodl.art/api/v1")

MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print(
    f"OpenAI key: {bool(OPENAI_API_KEY)} | Jev key: {bool(JEV_API_KEY)} | AutoDL key: {bool(AUTODL_API_KEY)}"
)

## Step 1 — Configure Experiment Scope, Backbones & Shared Persistent Clients

In [ ]:
# === CELL 3: Configure Mode, Backbones & Shared Clients ===
from dial_skill.benchmarks.when2speak import assert_no_calibration_leakage, load_when2speak_split
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache
from dial_skill.utils.checkpoint import save_run_metadata

MODE = "main"  # "main" (2,000 test turns) | "pilot" (400 test turns for quick verification)
QUALITY_N_SPEAK_TURNS = 100  # Stratified gold SPEAK turns for Utterance Quality evaluation
CROSS_MODEL_SPLIT = "pilot"  # 400 test turns across frontier & open-weight families

MAIN_BACKBONES = ["gpt-4.1-mini", "gpt-4.1"]
CROSS_FAMILY_BACKBONES = [
    {"id": "gpt-5.6-terra", "provider": "openai", "family": "OpenAI-5.6"},
    {"id": "gpt-4o", "provider": "openai", "family": "OpenAI-4o"},
    {"id": "DeepSeek-V4.1-Flash", "provider": "autodl", "family": "DeepSeek"},
    {"id": "GLM-5.3-flash", "provider": "autodl", "family": "GLM"},
]

RUN_DIR = DRIVE_ROOT / "runs" / "when2speak_v3_collab" / MODE
RUN_DIR.mkdir(parents=True, exist_ok=True)
save_run_metadata(RUN_DIR, {"mode": MODE, "main_backbones": MAIN_BACKBONES, "quality_n": QUALITY_N_SPEAK_TURNS})

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai_client = OpenAIClient(api_key=OPENAI_API_KEY, models_cfg=MODELS_CFG, cache=cache, provider_name="openai")
autodl_client = (
    OpenAIClient(
        api_key=AUTODL_API_KEY,
        models_cfg=MODELS_CFG,
        cache=cache,
        base_url=AUTODL_BASE_URL,
        provider_name="autodl",
        timeout_sec=120.0,
    )
    if AUTODL_API_KEY
    else None
)
triage_client = TriageClient(
    models_cfg=MODELS_CFG,
    cache=cache,
    openai_client=oai_client,
    luna_model="gpt-4.1-mini",
    jev_api_key=JEV_API_KEY,
)

val_turns = load_when2speak_split("val_calib", local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
eval_turns = load_when2speak_split(MODE, local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
assert_no_calibration_leakage(val_turns, eval_turns, max_calib_turns=200)
print(f"Loaded val_calib={len(val_turns)} turns (SPEAK={sum(t.label_speak for t in val_turns)}) | {MODE}={len(eval_turns)} turns (SPEAK={sum(t.label_speak for t in eval_turns)})")

## Step 2 — Equal Leak-Free Validation Calibration (`val_calib = 200` turns) + Held-Out Timing Benchmark

In [ ]:
# === CELL 4: Run & Calibrate All Baselines, Ablations & Closed-Loop DIAL-Collab ===
import pandas as pd
from dial_skill.baselines.direct_chat import DirectChatBaseline, DirectSelfConsistencyBaseline
from dial_skill.baselines.proact import ProACTBaseline
from dial_skill.baselines.single_gate import SingleGateBaseline
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.eval.metrics import (
    compute_when2speak_metrics,
    holm_bonferroni_correction,
    mcnemar_test,
    paired_bootstrap_test,
)
from dial_skill.eval.runners import (
    apply_1d_threshold_to_episodes,
    apply_dual_calibration_to_episodes,
    calibrate_1d_score_threshold,
    calibrate_platt_from_validation,
    calibrate_when2speak_thresholds,
    run_when2speak_eval,
    sweep_full_pareto_frontier,
)

def build_timing_agents(backbone: str, client: OpenAIClient):
    cfg_parallel = load_dial_skill_config(overrides={"ablation": {"gate_mode": "dual_margin", "disable_disagreement_deesc": True}})
    cfg_collab = DialSkillAgent.build_collab_config(enable_pre_delivery_audit=False, gate_mode="deliberative_collab")
    cfg_skep = load_dial_skill_config(overrides={"ablation": {"gate_mode": "skeptic_only", "disable_disagreement_deesc": True}})
    cfg_adv = load_dial_skill_config(overrides={"ablation": {"gate_mode": "advocate_only", "disable_disagreement_deesc": True}})
    cfg_comb = load_dial_skill_config(overrides={"ablation": {"gate_mode": "single_combined", "disable_disagreement_deesc": True}})

    return {
        "B0_direct": ("none", DirectChatBaseline(openai_client=client, model=backbone, method_name=f"B0_direct_{backbone}")),
        "B0_direct_sc3": ("1d", DirectSelfConsistencyBaseline(openai_client=client, model=backbone, n_samples=3, method_name=f"B0_direct_sc3_{backbone}")),
        "B1_single_llm": ("1d", SingleGateBaseline(openai_client=client, gate_model=backbone, gate_source="llm", method_name=f"B1_single_llm_{backbone}")),
        "B2_proact": ("1d", ProACTBaseline(openai_client=client, gate_model=backbone, method_name=f"B2_proact_{backbone}")),
        "A4_single_combined": ("2d", DialSkillAgent(triage_client=triage_client, openai_client=client, gate_model=backbone, executor_model=backbone, cfg=cfg_comb, method_name=f"A4_single_combined_{backbone}")),
        "A4_advocate_only": ("1d", DialSkillAgent(triage_client=triage_client, openai_client=client, gate_model=backbone, executor_model=backbone, cfg=cfg_adv, method_name=f"A4_advocate_only_{backbone}")),
        "A4_skeptic_only": ("1d", DialSkillAgent(triage_client=triage_client, openai_client=client, gate_model=backbone, executor_model=backbone, cfg=cfg_skep, method_name=f"A4_skeptic_only_{backbone}")),
        "DIAL_parallel": ("2d", DialSkillAgent(triage_client=triage_client, openai_client=client, gate_model=backbone, executor_model=backbone, cfg=cfg_parallel, method_name=f"DIAL_parallel_{backbone}")),
        "DIAL_Collab": ("2d", DialSkillAgent(triage_client=triage_client, openai_client=client, gate_model=backbone, executor_model=backbone, cfg=cfg_collab, method_name=f"DIAL_Collab_{backbone}")),
    }

y_eval = [t.label_speak for t in eval_turns]
all_timing_metrics = {}
all_calibrations = {}
eval_episodes_store = {}

# Also evaluate Jev-only single gate once (backbone-independent) calibrated on val_calib
jev_gate = SingleGateBaseline(openai_client=oai_client, gate_model="gpt-4.1-mini", triage_client=triage_client, gate_source="jev", method_name="B1_single_jev")
val_jev = await run_when2speak_eval(agent=jev_gate, turns=val_turns, output_jsonl=RUN_DIR / "val_B1_single_jev.jsonl", resume=True, generate_text=False)
cal_jev = calibrate_1d_score_threshold(val_turns=val_turns, val_episodes=val_jev, eval_turns=eval_turns)
raw_eval_jev = await run_when2speak_eval(agent=jev_gate, turns=eval_turns, output_jsonl=RUN_DIR / "eval_B1_single_jev.jsonl", resume=True, generate_text=False)
cal_val_jev = apply_1d_threshold_to_episodes(val_jev, cal_jev["best_threshold"])
cal_eval_jev = apply_1d_threshold_to_episodes(raw_eval_jev, cal_jev["best_threshold"])
platt_jev, platt_meta_jev = calibrate_platt_from_validation(val_turns=val_turns, val_episodes=cal_val_jev, eval_episodes=cal_eval_jev, eval_turns=eval_turns)
all_calibrations["B1_single_jev"] = {**cal_jev, **platt_meta_jev}
all_timing_metrics["B1_single_jev"] = {"method": "B1 Single-Gate (Jev, Cal.)", "model": "jev-latest", **compute_when2speak_metrics(y_eval, cal_eval_jev, calibrated_probs=platt_jev)}
eval_episodes_store["B1_single_jev"] = cal_eval_jev

for backbone in MAIN_BACKBONES:
    print(f"\n{'=' * 80}\nEvaluating & Calibrating When2Speak Suite on Backbone: {backbone}\n{'=' * 80}")
    agents = build_timing_agents(backbone, oai_client)
    for m_key, (cal_kind, agent) in agents.items():
        full_key = f"{m_key}__{backbone}"
        raw_eval = await run_when2speak_eval(
            agent=agent, turns=eval_turns, output_jsonl=RUN_DIR / f"eval_{full_key}.jsonl", resume=True, generate_text=False
        )
        if cal_kind == "none":
            cal_eval = raw_eval
            platt_probs = None
            all_calibrations[full_key] = {"calibration": "none_direct"}
        else:
            raw_val = await run_when2speak_eval(
                agent=agent, turns=val_turns, output_jsonl=RUN_DIR / f"val_{full_key}.jsonl", resume=True, generate_text=False
            )
            if cal_kind == "1d":
                cal_info = calibrate_1d_score_threshold(val_turns=val_turns, val_episodes=raw_val, eval_turns=eval_turns)
                cal_val = apply_1d_threshold_to_episodes(raw_val, cal_info["best_threshold"])
                cal_eval = apply_1d_threshold_to_episodes(raw_eval, cal_info["best_threshold"])
            else:
                cal_info = calibrate_when2speak_thresholds(
                    val_turns=val_turns, val_episodes=raw_val, eval_turns=eval_turns, theta_floor=0.25
                )
                cal_val = apply_dual_calibration_to_episodes(
                    raw_val, best_lambda=cal_info["best_lambda"], best_tau_dm=cal_info["best_tau_dm"], theta_floor=0.25
                )
                cal_eval = apply_dual_calibration_to_episodes(
                    raw_eval, best_lambda=cal_info["best_lambda"], best_tau_dm=cal_info["best_tau_dm"], theta_floor=0.25
                )
            platt_probs, platt_meta = calibrate_platt_from_validation(
                val_turns=val_turns, val_episodes=cal_val, eval_episodes=cal_eval, eval_turns=eval_turns
            )
            all_calibrations[full_key] = {**cal_info, **platt_meta}

        m_res = compute_when2speak_metrics(y_eval, cal_eval, calibrated_probs=platt_probs)
        all_timing_metrics[full_key] = {"method": m_key, "model": backbone, **m_res}
        eval_episodes_store[full_key] = cal_eval
        print(
            f"  [{full_key:34s}] MacroF1={m_res['macro_f1']:.3f} [{m_res['macro_f1_ci95'][0]:.3f},{m_res['macro_f1_ci95'][1]:.3f}] | "
            f"FIR={m_res['fir']:.3f} | MIR={m_res['mir']:.3f} | AUROC={m_res['auroc']:.3f} | ECE={m_res['ece']:.3f} | $/1k=${m_res['cost_per_1k_turns_usd']:.2f}"
        )

display(pd.DataFrame(list(all_timing_metrics.values())))

## Step 3 — Paired Bootstrap $\Delta\text{Macro-F1}$ Tests, Holm–Bonferroni Correction & Cross-Family Generalization

In [ ]:
# === CELL 5: Statistical Significance (Paired Bootstrap + Holm-Bonferroni) & Cross-Family Sweep ===
sig_raw_p = {}
sig_details = {}
for backbone in MAIN_BACKBONES:
    ours_preds = [e.gate.binary_speak for e in eval_episodes_store[f"DIAL_Collab__{backbone}"]]
    for base_name in ["B0_direct", "B0_direct_sc3", "B1_single_llm", "B2_proact", "A4_single_combined", "A4_advocate_only", "A4_skeptic_only", "DIAL_parallel"]:
        comp_key = f"DIAL_Collab_vs_{base_name}__{backbone}"
        base_preds = [e.gate.binary_speak for e in eval_episodes_store[f"{base_name}__{backbone}"]]
        pb = paired_bootstrap_test(y_eval, ours_preds, base_preds, n_bootstrap=5000, seed=42)
        mc = mcnemar_test(y_eval, ours_preds, base_preds)
        sig_raw_p[comp_key] = pb["p_value"]
        sig_details[comp_key] = {**pb, "mcnemar_p": mc["p_value"]}

holm_results = holm_bonferroni_correction(sig_raw_p, alpha=0.05)
for k, h in holm_results.items():
    sig_details[k].update(h)

print("Holm-Bonferroni Paired Bootstrap Significance Summary:")
display(pd.DataFrame.from_dict(sig_details, orient="index"))

# Cross-family generalization (calibrated per backbone on val_calib <= 200 turns!)
cross_turns = load_when2speak_split(CROSS_MODEL_SPLIT, local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
y_cross = [t.label_speak for t in cross_turns]
cross_rows = []
cross_pvals = {}

for spec in CROSS_FAMILY_BACKBONES:
    mid, prov, fam = spec["id"], spec["provider"], spec["family"]
    cli = autodl_client if prov == "autodl" else oai_client
    if cli is None:
        print(f"Skipping {mid} (AutoDL key not configured)")
        continue
    try:
        b0 = DirectChatBaseline(openai_client=cli, model=mid, method_name=f"B0_direct_{mid}")
        collab = DialSkillAgent(
            triage_client=triage_client,
            openai_client=cli,
            gate_model=mid,
            executor_model=mid,
            cfg=DialSkillAgent.build_collab_config(enable_pre_delivery_audit=False),
            method_name=f"DIAL_Collab_{mid}",
        )
        val_c = await run_when2speak_eval(agent=collab, turns=val_turns, output_jsonl=RUN_DIR / f"cross_val_{mid}.jsonl", resume=True, generate_text=False)
        cal_c = calibrate_when2speak_thresholds(val_turns=val_turns, val_episodes=val_c, eval_turns=cross_turns, theta_floor=0.20)
        ep_b0 = await run_when2speak_eval(agent=b0, turns=cross_turns, output_jsonl=RUN_DIR / f"cross_b0_{mid}.jsonl", resume=True, generate_text=False)
        raw_ep_col = await run_when2speak_eval(agent=collab, turns=cross_turns, output_jsonl=RUN_DIR / f"cross_collab_{mid}.jsonl", resume=True, generate_text=False)
        ep_col = apply_dual_calibration_to_episodes(raw_ep_col, best_lambda=cal_c["best_lambda"], best_tau_dm=cal_c["best_tau_dm"], theta_floor=0.20)

        m_b0 = compute_when2speak_metrics(y_cross, ep_b0)
        m_col = compute_when2speak_metrics(y_cross, ep_col)
        pb = paired_bootstrap_test(y_cross, [e.gate.binary_speak for e in ep_col], [e.gate.binary_speak for e in ep_b0], n_bootstrap=3000)
        cross_pvals[mid] = pb["p_value"]
        cross_rows.append({
            "family": fam,
            "model": mid,
            "direct_macro_f1": m_b0["macro_f1"],
            "dial_macro_f1": m_col["macro_f1"],
            "direct_fir": m_b0["fir"],
            "dial_fir": m_col["fir"],
            "dial_mir": m_col["mir"],
            "dial_auroc": m_col["auroc"],
            "bootstrap_p": pb["p_value"],
        })
        print(f"  [Cross {mid:20s}] Direct F1={m_b0['macro_f1']:.3f} -> DIAL-Collab F1={m_col['macro_f1']:.3f} (FIR={m_col['fir']:.3f}, MIR={m_col['mir']:.3f}, p={pb['p_value']:.4f})")
    except Exception as exc:
        print(f"  [WARN] Skipping {mid}: {exc}")

cross_holm = holm_bonferroni_correction(cross_pvals)
for row in cross_rows:
    row["holm_bootstrap_p"] = cross_holm[row["model"]]["holm_adjusted_p_value"]

## Step 4 — Goal 2: Utterance Quality Evaluation on Stratified `When2Speak` Gold `SPEAK` Turns (`N = 100`)

Evaluates **how well the agent speaks** when intervention is warranted, testing `B0 Direct`, `Solo Executor (w/o Pre-Delivery Audit)`, and **`DIAL-Collab (Draft → Skeptic-Auditor → Refine)`** against gold human reference replies with a **Cross-Family Blinded Dual-Judge** (`gpt-4.1` + `DeepSeek-V4.1-Flash`).

In [ ]:
# === CELL 6: Utterance Quality Benchmark (Draft -> Audit -> Refine vs Solo Executor vs Direct) ===
from dial_skill.eval.judge import BlindedDualJudge
from dial_skill.eval.metrics import paired_nonparametric_test
from dial_skill.schemas import GateDecision, InterventionTier
from dial_skill.utils.checkpoint import JsonlCheckpointWriter, load_jsonl

speak_pool = [t for t in eval_turns if t.label_speak and t.reference_response]
quality_turns = speak_pool[: min(QUALITY_N_SPEAK_TURNS, len(speak_pool))]
print(f"Selected {len(quality_turns)} stratified When2Speak SPEAK turns with gold reference utterances.")

dual_dialogue_judge = BlindedDualJudge(
    openai_client=oai_client,
    primary_judge_model="gpt-4.1",
    secondary_judge_model="DeepSeek-V4.1-Flash" if autodl_client is not None else "gpt-4.1",
    secondary_openai_client=autodl_client,
    seed=42,
)

utterance_rows = []
utterance_scores_by_key = {}

for backbone in MAIN_BACKBONES:
    # Force-speak gate configuration on ground-truth SPEAK turns so all 3 methods generate utterances on the exact same 100 turns!
    cfg_solo = load_dial_skill_config(
        overrides={
            "triage": {"theta_skip": -1.0},
            "gate": {"theta_floor": 0.0, "tau_dm": -2.0},
            "ablation": {"gate_mode": "advocate_only", "disable_pre_delivery_audit": True},
            "executor": {"enable_pre_delivery_audit": False},
        }
    )
    cfg_audited = load_dial_skill_config(
        overrides={
            "triage": {"theta_skip": -1.0},
            "gate": {"theta_floor": 0.0, "tau_dm": -2.0},
            "ablation": {"gate_mode": "advocate_only", "disable_pre_delivery_audit": False},
            "executor": {"enable_pre_delivery_audit": True},
        }
    )
    qual_agents = {
        "B0_direct": DirectChatBaseline(
            openai_client=oai_client,
            model=backbone,
            extra_memory_block=" An intervention IS warranted on this turn; write your best helpful reply (do NOT output SILENT).",
            method_name=f"Qual_B0_{backbone}",
        ),
        "Solo_Executor": DialSkillAgent(
            triage_client=triage_client,
            openai_client=oai_client,
            gate_model=backbone,
            executor_model=backbone,
            cfg=cfg_solo,
            method_name=f"Qual_SoloExec_{backbone}",
        ),
        "DIAL_Collab_Audited": DialSkillAgent(
            triage_client=triage_client,
            openai_client=oai_client,
            gate_model=backbone,
            executor_model=backbone,
            cfg=cfg_audited,
            method_name=f"Qual_DIAL_Audited_{backbone}",
        ),
    }

    for q_name, q_agent in qual_agents.items():
        q_key = f"{q_name}__{backbone}"
        q_eps = await run_when2speak_eval(
            agent=q_agent,
            turns=quality_turns,
            output_jsonl=RUN_DIR / f"utterance_eps_{q_key}.jsonl",
            resume=True,
            generate_text=True,
        )
        grade_path = RUN_DIR / f"utterance_grades_{q_key}.jsonl"
        existing_grades = {r["sample_id"]: r for r in load_jsonl(grade_path) if "sample_id" in r}
        g_writer = JsonlCheckpointWriter(grade_path)

        for turn, ep in zip(quality_turns, q_eps):
            if turn.sample_id in existing_grades:
                continue
            transcript = "\n".join(f"{m.speaker}: {m.text}" for m in turn.messages)
            cand_txt = (ep.response_text or "").strip() or "(remained silent)"
            g = await dual_dialogue_judge.grade_dialogue_response(
                sample_id=turn.sample_id,
                transcript=transcript,
                candidate_response=cand_txt,
                reference_response=turn.reference_response,
                method_tag_for_blinding=q_key,
            )
            g["word_count"] = len(cand_txt.split())
            g_writer.append(g)
            existing_grades[turn.sample_id] = g

        ordered_g = [existing_grades[t.sample_id] for t in quality_turns]
        utterance_scores_by_key[q_key] = [float(x["overall_score"]) for x in ordered_g]
        utterance_rows.append({
            "key": q_key,
            "method": q_name,
            "model": backbone,
            "overall_score": round(sum(x["overall_score"] for x in ordered_g) / len(ordered_g), 3),
            "helpfulness": round(sum(x["helpfulness"] for x in ordered_g) / len(ordered_g), 3),
            "conciseness": round(sum(x["conciseness"] for x in ordered_g) / len(ordered_g), 3),
            "appropriateness": round(sum(x["appropriateness"] for x in ordered_g) / len(ordered_g), 3),
            "avg_words": round(sum(x["word_count"] for x in ordered_g) / len(ordered_g), 1),
            "perm_p": None,
        })

for row in utterance_rows:
    b = row["model"]
    if row["method"] != "DIAL_Collab_Audited":
        p_res = paired_nonparametric_test(
            utterance_scores_by_key[f"DIAL_Collab_Audited__{b}"],
            utterance_scores_by_key[row["key"]],
        )
        row["perm_p"] = p_res["permutation_p_value"]

display(pd.DataFrame(utterance_rows))

## Step 5 — Export Publication LaTeX Tables, Pareto Frontier PDF & Complete JSON Summary

In [ ]:
# === CELL 7: Generate Publication LaTeX Tables & Continuous FIR-MIR Pareto Figure ===
from dial_skill.eval.report import (
    generate_tab_cross_model_when2speak,
    generate_tab_utterance_quality,
    generate_tab_when2speak,
    plot_fir_mir_pareto,
)

main_table_rows = list(all_timing_metrics.values())
tab_w2s_path = generate_tab_when2speak(
    main_table_rows, DRIVE_ROOT / "paper_artifacts" / "tables" / "tab_when2speak_v3_main.tex"
)
tab_qual_path = generate_tab_utterance_quality(
    utterance_rows, DRIVE_ROOT / "paper_artifacts" / "tables" / "tab_utterance_quality.tex"
)
if cross_rows:
    generate_tab_cross_model_when2speak(
        cross_rows, DRIVE_ROOT / "paper_artifacts" / "tables" / "tab_cross_model_when2speak_v3.tex"
    )

# Dense FIR-MIR Pareto frontier vs published When2Speak SFT/RL operating point
frontier_pts = sweep_full_pareto_frontier(
    turns=eval_turns,
    episodes=eval_episodes_store[f"DIAL_Collab__{MAIN_BACKBONES[-1]}"],
)
ref_pts = [
    {"label": "When2Speak SFT+RL (8B, trained)", "fir": 0.059, "mir": 0.514},
    {"label": "GPT-4o Zero-Shot (published)", "fir": 0.448, "mir": 0.251},
    {
        "label": f"B0 Direct ({MAIN_BACKBONES[0]})",
        "fir": all_timing_metrics[f"B0_direct__{MAIN_BACKBONES[0]}"]["fir"],
        "mir": all_timing_metrics[f"B0_direct__{MAIN_BACKBONES[0]}"]["mir"],
    },
    {
        "label": f"B2 ProACT Cal. ({MAIN_BACKBONES[0]})",
        "fir": all_timing_metrics[f"B2_proact__{MAIN_BACKBONES[0]}"]["fir"],
        "mir": all_timing_metrics[f"B2_proact__{MAIN_BACKBONES[0]}"]["mir"],
    },
]
fig_pareto_path = plot_fir_mir_pareto(
    pareto_points=frontier_pts,
    reference_points=ref_pts,
    out_path=DRIVE_ROOT / "paper_artifacts" / "figures" / "fig_fir_mir_pareto_v3.pdf",
)

bundle = {
    "mode": MODE,
    "calibrations": all_calibrations,
    "timing_metrics": all_timing_metrics,
    "holm_significance": sig_details,
    "cross_family_rows": cross_rows,
    "utterance_quality_rows": utterance_rows,
    "pareto_frontier": frontier_pts,
}
(RUN_DIR / "summary_when2speak_v3.json").write_text(json.dumps(bundle, indent=2), encoding="utf-8")
print("Saved tables & figures:", tab_w2s_path, tab_qual_path, fig_pareto_path)